# OptiTrack vs Mediapipe error by target dot / direction / circle

This notebook extends `optitrack_mediapipe_tracker_comparison.ipynb`.

Goal: instead of only one error number per finger, compute the tracker disagreement by commanded target location:

- circles/radii: 0, 1, 2, 3, 5, 8 cm
- directions: 8 compass directions
- fingers: index, middle, ring, pinky

## Important interpretation

The error used here is the already-calibrated 2D tracker error:

`error = distance(Mediapipe calibrated-to-OptiTrack XZ, OptiTrack XZ)`

So the output tells us where Mediapipe and OptiTrack disagree most, **after correcting zero point, axis alignment, scale, rotation, and time shift**.

Target location is inferred from the Mediapipe/session `object_x, object_y` command/target trace. Because the target moves continuously between dots, each sample is assigned to the nearest nominal radius/direction bin. A stricter `near_dot` flag is also saved for samples close to the nominal dot centers.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.')
MP_DIR = ROOT / 'Mediapipe-capture_handtest' / '2026_09_01_19_21_single_finger_config_fingers_motor_set_0_167'
COMPARE_DIR = ROOT / 'OT-results' / 'tracker_comparison'
OUT_DIR = COMPARE_DIR / 'per_target_error'
PLOT_DIR = OUT_DIR / 'plots'
SAMPLE_DIR = OUT_DIR / 'labeled_samples'
for d in [OUT_DIR, PLOT_DIR, SAMPLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEGMENT_METRICS_CSV = COMPARE_DIR / 'segment_error_metrics.csv'

TARGET_CENTER_X_PX = 320.0
TARGET_CENTER_Y_PX = 240.0
TARGET_RADII_CM = np.array([0, 1, 2, 3, 5, 8], dtype=float)
DIR8_DEG = np.arange(0, 360, 45, dtype=float)
DIR8_LABELS = np.array(['E', 'NE', 'N', 'NW', 'W', 'SW', 'S', 'SE'])

# Samples further than this from a direction/radius center are still assigned to nearest bin,
# but near_dot=False. Metrics below use all assigned samples by default and also save near-dot-only tables.
ANGLE_TOL_DEG = 17.5
RADIUS_TOL_CM = {
    0.0: 0.40,
    1.0: 0.35,
    2.0: 0.35,
    3.0: 0.50,
    5.0: 0.75,
    8.0: 1.00,
}

plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180
plt.rcParams['axes.grid'] = True

In [ ]:
def seconds(ts):
    ser = pd.Series(pd.to_datetime(ts))
    return ser.map(lambda x: x.value).to_numpy(dtype=float) / 1e9


def circular_angle_diff_deg(a, b):
    return np.abs((a - b + 180.0) % 360.0 - 180.0)


def infer_px_per_cm_for_pair(pair_name):
    """Infer target pixel scale from the known outer 8 cm ring.

    The target trace covers the radial protocol. Use the 99th percentile radius as the outer ring
    to avoid one-frame outliers, then divide by 8 cm.
    """
    tracking = pd.read_csv(MP_DIR / pair_name / 'tracking.csv')
    dx = tracking['object_x'].to_numpy(dtype=float) - TARGET_CENTER_X_PX
    # Image y increases downward. Use -dy so positive is visually North/up.
    dz_screen = -(tracking['object_y'].to_numpy(dtype=float) - TARGET_CENTER_Y_PX)
    radius_px = np.hypot(dx, dz_screen)
    outer_radius_px = np.nanpercentile(radius_px, 99.0)
    return float(outer_radius_px / 8.0)


def target_labels_from_object_xy(object_x, object_y, px_per_cm):
    dx = np.asarray(object_x, dtype=float) - TARGET_CENTER_X_PX
    z_screen = -(np.asarray(object_y, dtype=float) - TARGET_CENTER_Y_PX)
    radius_px = np.hypot(dx, z_screen)
    radius_cm_est = radius_px / px_per_cm

    # Nearest radius label.
    nearest_radius_idx = np.argmin(np.abs(radius_cm_est[:, None] - TARGET_RADII_CM[None, :]), axis=1)
    radius_label_cm = TARGET_RADII_CM[nearest_radius_idx]
    radius_error_cm = np.abs(radius_cm_est - radius_label_cm)

    # Nearest direction label. Center samples have no direction.
    angle_deg = (np.degrees(np.arctan2(z_screen, dx)) + 360.0) % 360.0
    nearest_dir_idx = np.argmin(circular_angle_diff_deg(angle_deg[:, None], DIR8_DEG[None, :]), axis=1)
    direction_label = DIR8_LABELS[nearest_dir_idx].astype(object)
    direction_deg = DIR8_DEG[nearest_dir_idx]
    direction_error_deg = circular_angle_diff_deg(angle_deg, direction_deg)

    is_center = radius_label_cm == 0
    direction_label[is_center] = 'CENTER'
    direction_deg[is_center] = np.nan
    direction_error_deg[is_center] = np.nan

    near_radius = np.array([radius_error_cm[i] <= RADIUS_TOL_CM[float(radius_label_cm[i])] for i in range(len(radius_label_cm))])
    near_angle = (direction_error_deg <= ANGLE_TOL_DEG) | is_center
    near_dot = near_radius & near_angle

    return pd.DataFrame({
        'target_dx_px': dx,
        'target_zscreen_px': z_screen,
        'target_radius_cm_est': radius_cm_est,
        'target_radius_cm': radius_label_cm,
        'target_radius_error_cm': radius_error_cm,
        'target_angle_deg_E0_CCW': angle_deg,
        'target_direction': direction_label,
        'target_direction_deg': direction_deg,
        'target_direction_error_deg': direction_error_deg,
        'near_dot': near_dot,
    })


def interpolate_object_xy(pair_name, query_epoch_s):
    tracking = pd.read_csv(MP_DIR / pair_name / 'tracking.csv')
    t = seconds(tracking['timestamp'])
    object_x = np.interp(query_epoch_s, t, tracking['object_x'].to_numpy(dtype=float))
    object_y = np.interp(query_epoch_s, t, tracking['object_y'].to_numpy(dtype=float))
    return object_x, object_y


def error_metrics(errors):
    e = np.asarray(errors, dtype=float)
    e = e[np.isfinite(e)]
    if len(e) == 0:
        return pd.Series({'n': 0, 'rmse_mm': np.nan, 'mean_error_mm': np.nan, 'p95_error_mm': np.nan, 'median_error_mm': np.nan})
    return pd.Series({
        'n': int(len(e)),
        'rmse_mm': float(np.sqrt(np.mean(e ** 2))),
        'mean_error_mm': float(np.mean(e)),
        'p95_error_mm': float(np.percentile(e, 95)),
        'median_error_mm': float(np.median(e)),
    })


def aggregate_metrics(df, group_cols, near_only=False):
    d = df.copy()
    if near_only:
        d = d[d['near_dot']].copy()
    d = d[np.isfinite(d['cv_error_mm'])].copy()
    if d.empty:
        return pd.DataFrame(columns=group_cols + ['n','rmse_mm','mean_error_mm','p95_error_mm','median_error_mm'])
    return d.groupby(group_cols, dropna=False)['cv_error_mm'].apply(error_metrics).unstack().reset_index()

In [ ]:
segment_metrics = pd.read_csv(SEGMENT_METRICS_CSV)
all_labeled = []
scale_rows = []

for _, seg in segment_metrics.iterrows():
    pair_name = seg['mp_pair']
    finger = seg['finger']
    sample_csv = Path(seg['segment_samples_csv'])
    samples = pd.read_csv(sample_csv)
    px_per_cm = infer_px_per_cm_for_pair(pair_name)
    scale_rows.append({'mp_pair': pair_name, 'finger': finger, 'target_px_per_cm_est': px_per_cm})

    object_x, object_y = interpolate_object_xy(pair_name, samples['mp_original_time_s'].to_numpy(dtype=float))
    labels = target_labels_from_object_xy(object_x, object_y, px_per_cm)

    labeled = pd.concat([samples, labels], axis=1)
    labeled.insert(0, 'finger', finger)
    labeled.insert(1, 'mp_pair', pair_name)
    labeled.insert(2, 'ot_file', seg['ot_file'])
    labeled['target_px_per_cm_est'] = px_per_cm

    out_csv = SAMPLE_DIR / f"{pair_name}_{finger}_{Path(sample_csv).stem}_labeled_targets.csv"
    labeled.to_csv(out_csv, index=False)
    all_labeled.append(labeled)

labeled_all = pd.concat(all_labeled, ignore_index=True)
scale_table = pd.DataFrame(scale_rows).drop_duplicates()
labeled_all.to_csv(OUT_DIR / 'all_labeled_error_samples.csv', index=False)
scale_table.to_csv(OUT_DIR / 'target_scale_estimates.csv', index=False)

# Metric tables.
by_finger_radius_dir = aggregate_metrics(labeled_all, ['finger', 'target_radius_cm', 'target_direction'])
by_finger_radius = aggregate_metrics(labeled_all, ['finger', 'target_radius_cm'])
by_finger_dir = aggregate_metrics(labeled_all[labeled_all['target_direction'] != 'CENTER'], ['finger', 'target_direction'])
by_radius_dir_all_fingers = aggregate_metrics(labeled_all, ['target_radius_cm', 'target_direction'])
by_finger_radius_dir_near = aggregate_metrics(labeled_all, ['finger', 'target_radius_cm', 'target_direction'], near_only=True)

# Stable ordering.
finger_order = {'index': 0, 'middle': 1, 'ring': 2, 'pinky': 3}
dir_order = {'E': 0, 'NE': 1, 'N': 2, 'NW': 3, 'W': 4, 'SW': 5, 'S': 6, 'SE': 7, 'CENTER': -1}
for table in [by_finger_radius_dir, by_finger_radius, by_finger_dir, by_finger_radius_dir_near]:
    if 'finger' in table:
        table['finger_order'] = table['finger'].map(finger_order).fillna(99)
    if 'target_direction' in table:
        table['direction_order'] = table['target_direction'].map(dir_order).fillna(99)

sort_cols = [c for c in ['finger_order', 'target_radius_cm', 'direction_order'] if c in by_finger_radius_dir]
by_finger_radius_dir = by_finger_radius_dir.sort_values(sort_cols).drop(columns=[c for c in ['finger_order','direction_order'] if c in by_finger_radius_dir])
by_finger_radius = by_finger_radius.sort_values([c for c in ['finger_order','target_radius_cm'] if c in by_finger_radius]).drop(columns=[c for c in ['finger_order'] if c in by_finger_radius])
by_finger_dir = by_finger_dir.sort_values([c for c in ['finger_order','direction_order'] if c in by_finger_dir]).drop(columns=[c for c in ['finger_order','direction_order'] if c in by_finger_dir])
by_finger_radius_dir_near = by_finger_radius_dir_near.sort_values([c for c in ['finger_order','target_radius_cm','direction_order'] if c in by_finger_radius_dir_near]).drop(columns=[c for c in ['finger_order','direction_order'] if c in by_finger_radius_dir_near])
by_radius_dir_all_fingers['direction_order'] = by_radius_dir_all_fingers['target_direction'].map(dir_order).fillna(99)
by_radius_dir_all_fingers = by_radius_dir_all_fingers.sort_values(['target_radius_cm','direction_order']).drop(columns=['direction_order'])

by_finger_radius_dir.to_csv(OUT_DIR / 'error_by_finger_radius_direction.csv', index=False)
by_finger_radius.to_csv(OUT_DIR / 'error_by_finger_radius.csv', index=False)
by_finger_dir.to_csv(OUT_DIR / 'error_by_finger_direction.csv', index=False)
by_radius_dir_all_fingers.to_csv(OUT_DIR / 'error_by_radius_direction_all_fingers.csv', index=False)
by_finger_radius_dir_near.to_csv(OUT_DIR / 'error_by_finger_radius_direction_near_dot_only.csv', index=False)

print('Estimated target pixel scale per pair:')
print(scale_table.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print('\nSamples by nearest radius/direction, all fingers:')
print(by_radius_dir_all_fingers[['target_radius_cm','target_direction','n','rmse_mm','mean_error_mm','p95_error_mm']].head(30).to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print('\nSaved tables to', OUT_DIR)

In [ ]:
# Heatmap helpers.
def plot_heatmap(table, value_col, title, out_path, finger=None):
    d = table.copy()
    if finger is not None:
        d = d[d['finger'].eq(finger)].copy()
    d = d[d['target_direction'] != 'CENTER'].copy()
    pivot = d.pivot_table(index='target_radius_cm', columns='target_direction', values=value_col, aggfunc='mean')
    pivot = pivot.reindex(index=[1,2,3,5,8], columns=['E','NE','N','NW','W','SW','S','SE'])

    fig, ax = plt.subplots(figsize=(9, 5.2))
    im = ax.imshow(pivot.to_numpy(dtype=float), aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{r:g} cm' for r in pivot.index])
    ax.set_xlabel('Direction')
    ax.set_ylabel('Circle / radius')
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(value_col.replace('_', ' '))

    for i, radius in enumerate(pivot.index):
        for j, direction in enumerate(pivot.columns):
            val = pivot.loc[radius, direction]
            if np.isfinite(val):
                ax.text(j, i, f'{val:.1f}', ha='center', va='center', color='white' if val > np.nanmean(pivot.to_numpy()) else 'black', fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)

# Per-finger heatmaps for mean/RMSE/p95.
for finger in ['index','middle','ring','pinky']:
    plot_heatmap(by_finger_radius_dir, 'mean_error_mm', f'{finger}: mean error by target circle/direction', PLOT_DIR / f'{finger}_mean_error_heatmap.png', finger=finger)
    plot_heatmap(by_finger_radius_dir, 'rmse_mm', f'{finger}: RMSE by target circle/direction', PLOT_DIR / f'{finger}_rmse_heatmap.png', finger=finger)
    plot_heatmap(by_finger_radius_dir, 'p95_error_mm', f'{finger}: 95th percentile error by target circle/direction', PLOT_DIR / f'{finger}_p95_error_heatmap.png', finger=finger)

plot_heatmap(by_radius_dir_all_fingers, 'mean_error_mm', 'All fingers: mean error by target circle/direction', PLOT_DIR / 'all_fingers_mean_error_heatmap.png')
plot_heatmap(by_radius_dir_all_fingers, 'rmse_mm', 'All fingers: RMSE by target circle/direction', PLOT_DIR / 'all_fingers_rmse_heatmap.png')
plot_heatmap(by_radius_dir_all_fingers, 'p95_error_mm', 'All fingers: 95th percentile error by target circle/direction', PLOT_DIR / 'all_fingers_p95_error_heatmap.png')

print('Wrote heatmaps to', PLOT_DIR)

In [ ]:
# Radial and direction summary plots.
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for finger, d in by_finger_radius.groupby('finger'):
    d = d.sort_values('target_radius_cm')
    ax.plot(d['target_radius_cm'], d['mean_error_mm'], marker='o', label=finger)
ax.set_xlabel('Target radius / circle (cm)')
ax.set_ylabel('Mean calibrated 2D error (mm)')
ax.set_title('Mean tracker error vs target circle')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'mean_error_by_radius_per_finger.png', bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5.2))
dir_order = ['E','NE','N','NW','W','SW','S','SE']
for finger, d in by_finger_dir.groupby('finger'):
    d = d.set_index('target_direction').reindex(dir_order).reset_index()
    ax.plot(d['target_direction'], d['mean_error_mm'], marker='o', label=finger)
ax.set_xlabel('Target direction')
ax.set_ylabel('Mean calibrated 2D error (mm)')
ax.set_title('Mean tracker error vs target direction')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'mean_error_by_direction_per_finger.png', bbox_inches='tight')
plt.close(fig)

print(PLOT_DIR / 'mean_error_by_radius_per_finger.png')
print(PLOT_DIR / 'mean_error_by_direction_per_finger.png')

In [ ]:
# Compact report: worst dots and interpretation.
min_n = 30
worst = by_finger_radius_dir[(by_finger_radius_dir['target_direction'] != 'CENTER') & (by_finger_radius_dir['n'] >= min_n)].sort_values('mean_error_mm', ascending=False).head(20)
center = by_finger_radius_dir[by_finger_radius_dir['target_direction'] == 'CENTER'].copy()

lines = []
lines.append('# Per-target OptiTrack vs Mediapipe error report')
lines.append('')
lines.append('## Method')
lines.append('- Used the already synchronized/calibrated OptiTrack-vs-Mediapipe comparison samples.')
lines.append('- Error metric is calibrated 2D X-Z positional disagreement in mm.')
lines.append('- Each sample was labeled by nearest commanded target radius/circle and 8-way direction from `object_x, object_y`.')
lines.append('- Center = `(320, 240)` pixels. Direction convention: E=+x, N=-image_y.')
lines.append('- Target pixel scale was inferred per Mediapipe pair from the outer 8 cm ring.')
lines.append('- Tables include all nearest-bin samples; a separate near-dot-only table is also saved.')
lines.append('')
lines.append('## Output tables')
lines.append('- `error_by_finger_radius_direction.csv`')
lines.append('- `error_by_finger_radius.csv`')
lines.append('- `error_by_finger_direction.csv`')
lines.append('- `error_by_radius_direction_all_fingers.csv`')
lines.append('- `error_by_finger_radius_direction_near_dot_only.csv`')
lines.append('- `all_labeled_error_samples.csv`')
lines.append('')
lines.append('## Worst target bins by mean error')
for _, r in worst.iterrows():
    lines.append(f"- {r['finger']} | {r['target_radius_cm']:g} cm | {r['target_direction']}: mean={r['mean_error_mm']:.2f} mm, RMSE={r['rmse_mm']:.2f} mm, p95={r['p95_error_mm']:.2f} mm, n={int(r['n'])}")
lines.append('')
lines.append('## Center bins')
if center.empty:
    lines.append('- No center samples found after labeling.')
else:
    for _, r in center.iterrows():
        lines.append(f"- {r['finger']}: center mean={r['mean_error_mm']:.2f} mm, RMSE={r['rmse_mm']:.2f} mm, p95={r['p95_error_mm']:.2f} mm, n={int(r['n'])}")

report = '\n'.join(lines)
(OUT_DIR / 'per_target_error_report.md').write_text(report, encoding='utf-8')
print(report)